In [4]:
#imports
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
import random
layers = tf.keras.layers
Model = tf.keras.Model

In [5]:

print(tf.__version__)
print(tf.config.list_physical_devices('GPU'))

2.19.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [6]:
print(tf.config.experimental.list_physical_devices('GPU'))
import subprocess
print(subprocess.run(['nvidia-smi', 'topo', '-m'], capture_output=True, text=True).stdout)

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks



In [7]:
strategy = tf.distribute.MirroredStrategy(cross_device_ops=tf.distribute.HierarchicalCopyAllReduce())
print(f"Number of devices: {strategy.num_replicas_in_sync}")

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of devices: 2


In [8]:
def load_and_resize(path1, path2, label):
    img1 = tf.io.read_file(path1)
    img2 = tf.io.read_file(path2)
    img1 = tf.image.decode_png(img1, channels=3)
    img2 = tf.image.decode_png(img2, channels=3)
    img1 = tf.image.resize(img1, (224, 224))
    img2 = tf.image.resize(img2, (224, 224))
    img1 = tf.cast(img1, tf.float32) / 255.0
    img2 = tf.cast(img2, tf.float32) / 255.0
    return (img1, img2), label

def augment_img(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.image.random_saturation(img, lower=0.85, upper=1.15)
    # keep values valid after brightness/contrast/saturation jitter
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img

def augment_pair(imgs, label):
    img1, img2 = imgs
    img1 = augment_img(img1)
    img2 = augment_img(img2)
    return (img1, img2), label

def create_dataset(pairs, batch_size=8, training=False):
    img1_paths = [p[0] for p in pairs]
    img2_paths = [p[1] for p in pairs]
    labels     = [p[2] for p in pairs]
    dataset = tf.data.Dataset.from_tensor_slices(
        ((img1_paths, img2_paths), labels)
    )
    options = tf.data.Options()
    options.experimental_deterministic = False
    dataset = dataset.with_options(options)

    # 1. Decode + resize (deterministic, safe to cache)
    dataset = dataset.map(
        lambda paths, label: load_and_resize(paths[0], paths[1], label),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    # 2. Cache the decoded/resized images (expensive disk I/O, never changes)
    dataset = dataset.cache()

    # 3. Shuffle
    dataset = dataset.shuffle(buffer_size=1000, reshuffle_each_iteration=True)

    # 4. Augment AFTER caching, so it's randomized fresh every epoch (training only)
    if training:
        dataset = dataset.map(augment_pair, num_parallel_calls=tf.data.AUTOTUNE)

    # 5. Batch
    dataset = dataset.batch(batch_size, drop_remainder=True)

    # 6. Prefetch for the GPU
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [9]:
import random

def make_pairs(char_dict, num_pairs=20):
    class_names = list(char_dict.keys())
    all_pairs = []
    for current_class in class_names:
        images = char_dict[current_class]
        if len(images) < 2:
            continue  # skip species with too few images to form a same-pair

        # same pairs (label = 1)
        used_same = set()
        attempts = 0
        max_attempts = num_pairs * 20  # safety valve for small image counts
        while len(used_same) < num_pairs and attempts < max_attempts:
            img1 = random.choice(images)
            img2 = random.choice(images)
            attempts += 1
            if img2 == img1:
                continue
            pair_check = tuple(sorted([img1, img2]))
            if pair_check in used_same:
                continue
            used_same.add(pair_check)
            all_pairs.append([img1, img2, 1])

        # different pairs (label = 0), drawing class2 from a wide pool each time
        used_diff = set()
        attempts = 0
        while len(used_diff) < num_pairs and attempts < max_attempts:
            class2 = random.choice([x for x in class_names if x != current_class])
            img1 = random.choice(images)
            img2 = random.choice(char_dict[class2])
            attempts += 1
            pair_check = tuple(sorted([img1, img2]))
            if pair_check in used_diff:
                continue
            used_diff.add(pair_check)
            all_pairs.append([img1, img2, 0])

    random.shuffle(all_pairs)
    return all_pairs

In [10]:
layers = tf.keras.layers
Model = tf.keras.Model
regularizers = tf.keras.regularizers

def build_encoder(input_shape=(224, 224, 3), embedding_dim=128, freeze_backbone=False, l2_reg=1e-4):
    inputs = layers.Input(shape=input_shape, name="image")
    base_model = tf.keras.applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape,
        pooling='avg'
    )
    base_model.trainable = not freeze_backbone
    x = base_model(inputs)
    x = layers.Dense(512, activation="relu", kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(embedding_dim, kernel_regularizer=regularizers.l2(l2_reg))(x)
    embeddings = layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1))(x)
    return Model(inputs, embeddings)

def build_siamese_model(enc):
    img1 = layers.Input(shape=(224, 224, 3), name='img1')
    img2 = layers.Input(shape=(224, 224, 3), name='img2')
    tensor1 = enc(img1)
    tensor2 = enc(img2)
    distance = layers.Lambda(lambda tensors: tf.norm(tensors[0] - tensors[1], axis=1))([tensor1, tensor2])
    return Model([img1, img2], distance)

In [ ]:
char_dict = load_character_images('/kaggle/input/datasets/qweenink/omniglot/images_background_small1/images_background_small1')
pairs = make_pairs(char_dict)
dataset = create_dataset(pairs, batch_size=64)

dataset = strategy.experimental_distribute_dataset(dataset)

with strategy.scope():
    enc = build_encoder(freeze_backbone=False)
    siamese = build_siamese_model(enc)
    optimizer = tf.keras.optimizers.Adam(1e-5)

print(f"Total pairs: {len(pairs)}")
print(f"Sample pair: {pairs[0]}")

In [19]:
def contrastive_loss(labels, distances, margin=1.0):
    labels = tf.cast(labels, tf.float32)
    same = labels * tf.square(distances)
    diff = (1 - labels) * tf.square(tf.maximum(margin - distances, 0))
    return same + diff

@tf.function
def train_step(batch):
    (img1, img2), labels = batch

    with tf.GradientTape() as tape:
        distances = siamese([img1, img2], training=True)

        loss = contrastive_loss(labels, distances, margin=1.0)

        loss = tf.nn.compute_average_loss(
            loss,
            global_batch_size=GLOBAL_BATCH_SIZE
        )

    grads = tape.gradient(loss, siamese.trainable_variables)
    optimizer.apply_gradients(zip(grads, siamese.trainable_variables))

    return loss


@tf.function
def distributed_train_step(batch):
    per_replica_losses = strategy.run(train_step, args=(batch,))

    return strategy.reduce(
        tf.distribute.ReduceOp.SUM,
        per_replica_losses,
        axis=None
    )


In [ ]:
epochs = 10
epoch_losses = []

best_loss = float("inf")
patience = 3
no_improve = 0

GLOBAL_BATCH_SIZE = 64   # change if you changed batch_size



for epoch in range(epochs):
    batch_losses = []

    for batch in dataset:
        loss = distributed_train_step(batch)
        batch_losses.append(loss.numpy())

    epoch_loss = np.mean(batch_losses)
    epoch_losses.append(epoch_loss)

    print(f"Epoch {epoch + 1}, Loss: {epoch_loss:.6f}")

    siamese.save(f"/kaggle/working/siamese_epoch{epoch+1}.keras")

    if epoch_loss < best_loss - 1e-5:
        best_loss = epoch_loss
        no_improve = 0
        enc.save("/kaggle/working/encoder_best.keras")
        print(f"  -> Best encoder saved (loss={best_loss:.6f})")
    else:
        no_improve += 1
        print(f"  -> No improvement ({no_improve}/{patience})")

        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break


enc.save("/kaggle/working/encoder_final.keras")

plt.figure(figsize=(6,4))
plt.plot(epoch_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Curve")
plt.grid(True)

plt.savefig(
    "/kaggle/working/training_curve.png",
    dpi=150,
    bbox_inches="tight"
)
plt.close()

print("Done")

In [ ]:
enc.save("/kaggle/working/encoder_final.keras")
plt.figure(figsize=(6,4))
plt.plot(epoch_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Curve")
plt.grid(True)

plt.savefig(
    "/kaggle/working/training_curve.png",
    dpi=150,
    bbox_inches="tight"
)
plt.close()

## cub200 retraining

In [ ]:
# Create a folder in your writable working directory
!mkdir -p /kaggle/working/cub200

# Extract the .tgz file from the input directory into the working directory
!tar -xzf /kaggle/input/datasets/hridayeshrana/cub-200-2011/CUB_200_2011.tgz -C /kaggle/working/cub200

In [11]:
import os
import random
from sklearn.model_selection import train_test_split

def load_and_split_cub_images(images_dir, test_size=0.2, val_size=0.1):
    """
    Loads CUB-200 images and splits species into train/val/test sets.
    val_size is the fraction of the *original* total reserved for validation
    (taken out of the remaining train portion after the test split).
    """
    all_species = [d for d in os.listdir(images_dir) if os.path.isdir(os.path.join(images_dir, d))]

    # First split off the test species
    train_val_species, test_species = train_test_split(
        all_species, test_size=test_size, random_state=42
    )

    # Then split the remainder into train/val
    # Adjust val_size so it represents the right fraction of the original whole
    relative_val_size = val_size / (1 - test_size)
    train_species, val_species = train_test_split(
        train_val_species, test_size=relative_val_size, random_state=42
    )

    def get_images_for_species(species_list):
        data = {}
        for species in species_list:
            species_path = os.path.join(images_dir, species)
            full_paths = [
                os.path.join(species_path, f)
                for f in os.listdir(species_path)
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))
            ]
            if full_paths:
                data[species] = full_paths
        return data

    return (
        get_images_for_species(train_species),
        get_images_for_species(val_species),
        get_images_for_species(test_species),
    )

# 1. Load and Split
cub_path = '/kaggle/working/cub200/CUB_200_2011/images'
train_dict, val_dict, test_dict = load_and_split_cub_images(cub_path, test_size=0.2, val_size=0.1)
print(f"Species for training: {len(train_dict)}, validation: {len(val_dict)}, testing: {len(test_dict)}")

# 2. Generate Pairs
train_pairs = make_pairs(train_dict, num_pairs=40)
val_pairs = make_pairs(val_dict, num_pairs=10)
test_pairs = make_pairs(test_dict, num_pairs=10)  # Fewer pairs for testing

# 3. Create Datasets
train_dataset = create_dataset(train_pairs, batch_size=16, training=True)
val_dataset = create_dataset(val_pairs, batch_size=16, training=False)
test_dataset = create_dataset(test_pairs, batch_size=16, training=False)

# 4. Distribute
train_dataset = strategy.experimental_distribute_dataset(train_dataset)
val_dataset = strategy.experimental_distribute_dataset(val_dataset)
test_dataset = strategy.experimental_distribute_dataset(test_dataset)

print(f"Ready: {len(train_pairs)} training pairs, {len(val_pairs)} validation pairs, {len(test_pairs)} test pairs.")

Species for training: 140, validation: 20, testing: 40
Ready: 11200 training pairs, 400 validation pairs, 800 test pairs.


In [12]:
with strategy.scope():
    enc = build_encoder()

    enc.load_weights('/kaggle/working/encoder_final.keras')
    # Rebuild the Siamese wrapper around your pre-trained encoder
    siamese = build_siamese_model(enc)
    
    # Initial learning rate for fine-tuning
    initial_lr = 1e-5
    optimizer = tf.keras.optimizers.Adam(initial_lr) 

    enc.summary()
    siamese.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 2048)           │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 128)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,801,024 (94.61 MB)

 Trainable params: 24,747,904 (94.41 MB)

...

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ img1 (InputLayer)   │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ img2 (InputLayer)   │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ (None, 128)       │ 24,801,024 │ img1[0][0],       │
│ (Functional)        │                   │            │ img2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None)            │          0 │ functional[0][0], │
│                     │                   │            │ functional[1][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,801,024 (94.61 MB)

 Trainable params: 24,747,904 (94.41 MB)

 Non-trainable params: 53,120 (207.50 KB)

In [ ]:
import numpy as np
from tqdm.auto import tqdm

# 1. Define the loss function
def contrastive_loss(labels, distances, margin=1.0):
    labels = tf.cast(labels, tf.float32)
    same = labels * tf.square(distances)
    diff = (1 - labels) * tf.square(tf.maximum(margin - distances, 0))
    return same + diff

# 2. Define the core training step (runs on a single device)
@tf.function
def train_step(batch):
    (img1, img2), labels = batch
    with tf.GradientTape() as tape:
        distances = siamese([img1, img2], training=True)
        loss = tf.reduce_mean(contrastive_loss(labels, distances))

    grads = tape.gradient(loss, siamese.trainable_variables)
    optimizer.apply_gradients(zip(grads, siamese.trainable_variables))
    return loss

# 2b. Define the core validation step (no gradient updates)
@tf.function
def val_step(batch):
    (img1, img2), labels = batch
    distances = siamese([img1, img2], training=False)
    loss = tf.reduce_mean(contrastive_loss(labels, distances))
    return loss

# 3. Define the distributed wrappers (unwraps PerReplica data)
@tf.function
def distributed_train_step(batch):
    per_replica_losses = strategy.run(train_step, args=(batch,))
    return strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica_losses, axis=None)

@tf.function
def distributed_val_step(batch):
    per_replica_losses = strategy.run(val_step, args=(batch,))
    return strategy.reduce(tf.distribute.ReduceOp.MEAN, per_replica_losses, axis=None)

# 4. Training loop
initial = 5
best_val_loss = float("inf")
patience_counter = 0

# NEW: lists to store average loss per epoch, for later plotting
train_loss_history = []
val_loss_history = []

print("Starting Training...")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    # --- Training ---
    batch_losses = []
    train_pbar = tqdm(train_dataset, desc="  Training", unit="batch")
    for batch in train_pbar:
        loss = distributed_train_step(batch)
        loss_val = loss.numpy()
        batch_losses.append(loss_val)
        train_pbar.set_postfix(loss=f"{loss_val:.4f}")
    avg_train_loss = np.mean(batch_losses)
    train_loss_history.append(avg_train_loss)  # NEW

    # --- Validation ---
    val_batch_losses = []
    val_pbar = tqdm(val_dataset, desc="  Validation", unit="batch")
    for batch in val_pbar:
        val_loss = distributed_val_step(batch)
        val_loss_val = val_loss.numpy()
        val_batch_losses.append(val_loss_val)
        val_pbar.set_postfix(loss=f"{val_loss_val:.4f}")
    avg_val_loss = np.mean(val_batch_losses)
    val_loss_history.append(avg_val_loss)  # NEW

    print(f"Epoch {epoch + 1}/{num_epochs} Summary -> Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")

    # Checkpointing and LR Reduction logic (based on validation loss)
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        siamese.save("/kaggle/working/best_model.keras")
        print("  -> Improvement! Saved best model.")
    else:
        patience_counter += 1
        print(f"  -> No improvement ({patience_counter}/3)")
        if patience_counter >= 3:
            new_lr = float(optimizer.learning_rate.numpy()) * 0.5
            optimizer.learning_rate.assign(new_lr)
            patience_counter = 0
            print(f"  -> Reducing LR to {new_lr:.2e}")

print("\nTraining complete.")

Starting Training...

Epoch 1/30


  Training: 0batch [00:00, ?batch/s]

INFO:tensorflow:batch_all_reduce: 218 all-reduces with algorithm = hierarchical_copy, num_packs = 1
INFO:tensorflow:batch_all_reduce: 218 all-reduces with algorithm = hierarchical_copy, num_packs = 1
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


I0000 00:00:1782888406.510844     101 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1782888407.412848     104 cuda_dnn.cc:529] Loaded cuDNN version 91002


  Validation: 0batch [00:00, ?batch/s]

...

  Training: 0batch [00:00, ?batch/s]

  Validation: 0batch [00:00, ?batch/s]

Epoch 16/30 Summary -> Train Loss: 0.114543, Val Loss: 0.183285
  -> No improvement (1/3)

Epoch 17/30


  Training: 0batch [00:00, ?batch/s]

KeyboardInterrupt: 

In [24]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_loss_history) + 1), train_loss_history, label="Train Loss")
plt.plot(range(1, len(val_loss_history) + 1), val_loss_history, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss 2")
plt.legend()
plt.grid(True)
plt.savefig(
    "/kaggle/working/val_vs_train.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

In [26]:
import json

# Extract just the species names (the keys) for the split record
split_record = {
    "train_species": list(train_dict.keys()),
    "val_species": list(val_dict.keys()),
    "test_species": list(test_dict.keys())
}

# Save to a JSON file in your working directory
with open("/kaggle/working/cub200_dataset_splits.json", "w") as f:
    json.dump(split_record, f, indent=4)
    
print("Splits successfully saved to JSON!")


Splits successfully saved to JSON!
